# Lab 2｜训练 + full-ranking 评估

Lab 1 最后把你的代码导出成了 `labs/myrec/model.py`。这一章接着用那份代码：在真实 MovieLens 上训练一轮，然后写一套评估，回答一个训练 loss 回答不了的问题——推荐到底准不准。

符号沿用 Lab 1：📖 概念，🧩 要你写的代码，✅ 断言，🚀 可以改着玩的地方，💡 折叠的参考答案。

前提是 Lab 1 的最后一格跑过了。没跑过的话，下面第一格会直接告诉你。

In [ ]:
# 不用改：接上 Lab 1 导出的模块
import math
import sys
from collections import defaultdict
from pathlib import Path

import torch


def find_labs_dir() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (base, base / 'labs'):
            if (candidate / 'labkit.py').exists():
                return candidate
    raise FileNotFoundError('找不到 labs 目录，请从仓库根目录或 labs 目录启动')


LABS_DIR = find_labs_dir()
if str(LABS_DIR) not in sys.path:
    sys.path.insert(0, str(LABS_DIR))

import labkit  # noqa: E402

try:
    from myrec.model import TwoTower, in_batch_cross_entropy_loss, train_one_step
except ModuleNotFoundError as error:
    raise SystemExit('先跑完 Lab 1：它最后一格会把代码导出成 labs/myrec/model.py') from error

PASSED = []


def check(name: str, actual, expected) -> None:
    '''严格相等：列表、元组、整数、布尔。'''
    ok = actual == expected
    print(('PASS' if ok else 'FAIL'), name, '| 期望:', expected, '| 实际:', actual)
    if not ok:
        raise AssertionError('还没对，改那一格，或看下面的参考答案')
    PASSED.append(name)


def check_close(name: str, actual: float, expected: float, tolerance: float = 1e-5) -> None:
    '''浮点比较，允许误差。'''
    ok = abs(actual - expected) <= tolerance
    print(('PASS' if ok else 'FAIL'), name, '| 期望约:', expected, '| 实际:', actual)
    if not ok:
        raise AssertionError('还没对，改那一格，或看下面的参考答案')
    PASSED.append(name)


print('labs dir:', LABS_DIR)
print('TwoTower 来自:', TwoTower.__module__)

In [ ]:
# 不用改：真实数据 + DataLoader
labkit.set_seed(42)
data = labkit.prepare_movielens()
print(data.describe())

train_loader = labkit.create_train_loader(data.train, batch_size=256, seed=42)
print('训练 batch 数:', len(train_loader))

## 📖 理解一下：一个 epoch 就是重复

Lab 1 写的 `train_one_step` 已经完成了「取一个 batch → forward → 算 loss → 反向 → 更新」。

一个 epoch 没有任何新东西，就是把它在所有 batch 上跑一遍，顺便把每一步的 loss 记下来求平均。

### 🧩 你来写：`train_one_epoch()`

`max_batches` 用来只跑前几个 batch，方便快速检查；传 `None` 就跑完整轮。

In [ ]:
def train_one_epoch(model, dataloader, optimizer, max_batches=None):
    '''训练一个 epoch，返回这一轮的平均 loss（float）。'''
    # TODO 2-0：遍历 dataloader，每个 batch 调用一次 train_one_step
    #   进循环之前先切到训练模式（model.train()）
    #   max_batches 不为 None 时，跑到第 max_batches 个 batch 就停
    #   返回所有 step loss 的平均值
    raise NotImplementedError('TODO 2-0')

In [ ]:
# ✅ 跑一下：先用 30 个 batch 确认它能跑，且真的在更新参数
labkit.set_seed(42)
probe_model = TwoTower(data.num_users, data.num_items, embedding_dim=64, temperature=0.1)
probe_optimizer = torch.optim.Adam(probe_model.parameters(), lr=1e-2)

before = probe_model.user_embedding.weight.detach().clone()
probe_loss = train_one_epoch(probe_model, train_loader, probe_optimizer, max_batches=30)

check('2-0 返回 float', isinstance(probe_loss, float), True)
check('2-0 loss 是有限值', bool(math.isfinite(probe_loss)), True)
check(
    '2-0 参数被更新了',
    bool(torch.equal(before, probe_model.user_embedding.weight.detach())),
    False,
)
print()
print('30 个 batch 的平均 loss =', round(probe_loss, 4))
print('参考：随机初始化时接近 ln(256) =', round(math.log(256), 4))

<details>
<summary>💡 参考答案（<code>train_one_epoch</code>）</summary>

```python
def train_one_epoch(model, dataloader, optimizer, max_batches=None):
    '''训练一个 epoch，返回这一轮的平均 loss（float）。'''
    model.train()
    losses = []
    for batch_number, batch in enumerate(dataloader):
        if max_batches is not None and batch_number >= max_batches:
            break
        losses.append(train_one_step(model, batch, optimizer))
    return sum(losses) / len(losses)
```

</details>

`model.train()` 放在 epoch 循环开头，而不是塞进每个 step 里。它决定 Dropout、BatchNorm 这些层的行为——`recommend_top_k` 里调过 `model.eval()`，不切回来就会一直停在推理模式。
如果你想上 GPU，在这一格里对 `batch` 的每个 tensor 调一次 `.to(device)`，并让 model 也在同一个 device 上——本 lab 统一用 CPU。

## 📖 理解一下：训练 loss 不是推荐质量

loss 降了，只能说明模型在优化我们写下的那个目标，它不回答「推荐准不准」。

更具体一点：训练时每一行的候选是同一个 batch 里的 256 个物品；而用户真正面对的是全部 3,533 部电影。候选集不一样，数字之间没有可比性。

所以接下来得自己写一套评估。这也是离线实验里最容易被跳过、又最容易出错的一步。

## 📖 理解一下：评估要做三件事

对每个待评测的用户：

1. 拿到他对**全部物品**的分数，取最高的 K 个；
2. 取之前，把他训练时已经看过的物品分数压成 `-inf`；
3. 看他的 target（留出来的那条交互）有没有落在这 K 个里。

第 2 步不能省。模型在训练集上见过这些物品，如果还允许把它们推出来，随便一个记住了训练集的模型都能刷到很高的分，这个数就没有意义了。

先写第 2 步需要的数据结构。

> 完整推导与文献：[03｜loss 会降，推荐准不准？](../docs/01-two-tower/03-loss-vs-recall.md)

### 🧩 你来写：`group_seen_items()`

把 train 交互整理成 `{user_idx: {item_idx, ...}}`，方便按用户查。

In [ ]:
def group_seen_items(interactions):
    '''把交互列表整理成 {user_idx: {item_idx, ...}}。'''
    # TODO 2-1：提示 defaultdict(set)
    raise NotImplementedError('TODO 2-1')

In [ ]:
# ✅ 跑一下：先用手造的三条交互验证
sample = [
    labkit.Interaction(user_idx=0, item_idx=7, timestamp=1),
    labkit.Interaction(user_idx=0, item_idx=9, timestamp=2),
    labkit.Interaction(user_idx=3, item_idx=7, timestamp=3),
]
sample_seen = group_seen_items(sample)

check('2-1 用户 0 看过两个', sorted(sample_seen[0]), [7, 9])
check('2-1 用户 3 看过一个', sorted(sample_seen[3]), [7])

<details>
<summary>💡 参考答案（<code>group_seen_items</code>）</summary>

```python
def group_seen_items(interactions):
    '''把交互列表整理成 {user_idx: {item_idx, ...}}。'''
    seen = defaultdict(set)
    for row in interactions:
        seen[row.user_idx].add(row.item_idx)
    return dict(seen)
```

</details>

返回 `dict(...)` 而不是直接返回 `defaultdict`：后面查询时用 `seen.get(user, set())`，不需要「查不到就自动建一个空集合」的副作用。

## 📖 理解一下：full-ranking 怎么算

物品向量只跟物品有关，可以一次性全部编码好，得到 `[I, D]`；用户向量按批算，得到 `[Q, D]`。两者一乘就是 `[Q, I]` —— 每行一个用户对全部物品的分数。

MovieLens 只有 3,533 个物品，直接算满就够，不需要向量检索库。这也是为什么第一步就强调 item 侧可以提前算：换成上亿候选时，省的就是这一步。

### 🧩 你来写：`recommend_top_k()`

返回 `{user_idx: [item_idx, ...]}`，每个用户 K 个。评分阶段不需要梯度，记得关掉。

In [ ]:
def recommend_top_k(model, user_indices, num_items, seen_items, k, user_batch_size=512):
    '''返回 {user_idx: [item_idx, ...]}，每个用户 k 个。'''
    # TODO 2-2：四步
    #   1) 编码全部物品 -> [I, D]
    #   2) 用户按批编码 -> [Q, D]，和物品矩阵相乘 -> [Q, I]
    #   3) 把该用户已看物品对应的分数设成 -inf
    #   4) torch.topk 取前 k 个，收集成 dict
    #   提示：用 @torch.no_grad() 装饰整个函数
    raise NotImplementedError('TODO 2-2')

In [ ]:
# ✅ 跑一下：20 个物品的小模型上检查形状和过滤
small_model = TwoTower(10, 20, embedding_dim=8, temperature=0.1)
small_seen = {0: {1, 2, 3}}
small_recs = recommend_top_k(
    small_model, [0, 1, 2], num_items=20, seen_items=small_seen, k=5
)

check('2-2 每个用户返回 5 个', [len(small_recs[user]) for user in (0, 1, 2)], [5, 5, 5])
check('2-2 已看物品没被推荐', sorted(set(small_recs[0]) & small_seen[0]), [])
check('2-2 五个推荐互不重复', len(set(small_recs[0])), 5)
check('2-2 返回的 item_idx 都在合法范围内', all(0 <= item < 20 for item in small_recs[0]), True)

<details>
<summary>💡 参考答案（<code>recommend_top_k</code>）</summary>

```python
@torch.no_grad()
def recommend_top_k(model, user_indices, num_items, seen_items, k, user_batch_size=512):
    '''返回 {user_idx: [item_idx, ...]}，每个用户 k 个。'''
    model.eval()
    all_item_ids = torch.arange(num_items)
    all_item_vectors = model.encode_items(all_item_ids)          # [I, D]

    recommendations = {}
    for start in range(0, len(user_indices), user_batch_size):
        batch_users = list(user_indices[start:start + user_batch_size])
        user_ids = torch.tensor(batch_users)
        scores = model.encode_users(user_ids) @ all_item_vectors.T   # [Q, I]

        for row, user_idx in enumerate(batch_users):
            already_seen = seen_items.get(user_idx, set())
            if already_seen:
                scores[row, torch.tensor(sorted(already_seen))] = -torch.inf

        top_items = torch.topk(scores, k=k, dim=1).indices.tolist()
        recommendations.update(zip(batch_users, top_items))
    return recommendations
```

</details>

两个容易踩的点：`model.eval()` 要在编码之前调用（这里没有 BatchNorm/Dropout，影响不大，但养成习惯）；`topk` 返回的是 `(values, indices)` 两个张量，别把 values 当成结果。

`user_batch_size` 控制一次算多少个用户。`[Q, I]` 这个分数矩阵的大小是 `Q × 3533`，用户多的时候要分批，不然内存会顶不住。

## 📖 理解一下：Recall@K 和 NDCG@K

每个评测用户只留了一个 target（他最后一次交互）。所以：

- **Recall@K**：target 落进前 K 个就算 1 分，否则 0 分，最后对所有用户求平均。
- **NDCG@K**：还看命中位置。排在第 `r` 位（从 0 开始数）贡献 `1 / log2(r + 2)`。只有一个 target 时，理想的归一化因子正好是 1，所以直接累加再平均就行。
- 单 target 协议下 HitRate@K 和 Recall@K 数值相同；只有多 target 时两者才会分开。

> 完整推导与文献：[03｜loss 会降，推荐准不准？](../docs/01-two-tower/03-loss-vs-recall.md)

### 🧩 你来写：`recall_at_k()` 和 `ndcg_at_k()`

两个函数都是「遍历所有 target、累加、除以用户数」。`targets` 是 `Interaction` 的列表，每条对应用户的 target 物品。

In [ ]:
def recall_at_k(recommendations, targets, k):
    '''返回 Recall@k（float，0~1）。'''
    # TODO 2-3：target 落进前 k 个就记一分
    raise NotImplementedError('TODO 2-3')


def ndcg_at_k(recommendations, targets, k):
    '''返回 NDCG@k（float，0~1）。命中时按排名折损：1 / log2(rank + 2)。'''
    # TODO 2-4：rank 从 0 开始数
    raise NotImplementedError('TODO 2-4')

In [ ]:
# ✅ 跑一下：三个用户的例子，数字能手算出来
fake_targets = [
    labkit.Interaction(user_idx=0, item_idx=5, timestamp=1),    # 排第 0 位
    labkit.Interaction(user_idx=1, item_idx=5, timestamp=1),    # 排第 2 位
    labkit.Interaction(user_idx=2, item_idx=99, timestamp=1),   # 没命中
]
fake_recs = {0: [5, 1, 2], 1: [1, 2, 5], 2: [1, 2, 3]}

check_close('2-3 Recall@3 = 2/3', recall_at_k(fake_recs, fake_targets, 3), 2 / 3)
check_close('2-3 Recall@1 = 1/3', recall_at_k(fake_recs, fake_targets, 1), 1 / 3)
# 两个命中：1 / log2(0 + 2) = 1，1 / log2(2 + 2) = 0.5
check_close('2-4 NDCG@3 = (1 + 0.5) / 3', ndcg_at_k(fake_recs, fake_targets, 3), 1.5 / 3)
check_close('2-4 没命中时是 0', ndcg_at_k(fake_recs, fake_targets[2:], 1), 0.0)

<details>
<summary>💡 参考答案（<code>recall_at_k</code> / <code>ndcg_at_k</code>）</summary>

```python
def recall_at_k(recommendations, targets, k):
    '''返回 Recall@k（float，0~1）。'''
    hits = 0
    for row in targets:
        if row.item_idx in recommendations[row.user_idx][:k]:
            hits += 1
    return hits / len(targets)


def ndcg_at_k(recommendations, targets, k):
    '''返回 NDCG@k（float，0~1）。命中时按排名折损：1 / log2(rank + 2)。'''
    total = 0.0
    for row in targets:
        top_k = recommendations[row.user_idx][:k]
        if row.item_idx in top_k:
            rank = top_k.index(row.item_idx)
            total += 1.0 / math.log2(rank + 2)
    return total / len(targets)
```

</details>

`rank + 2` 里的 `+2`：`rank` 从 0 开始，而 `log2(1) = 0` 会让第一名的权重变成无穷大，所以分母是 `rank + 2`——第一名贡献 `1 / log2(2) = 1`，第二名 `1 / log2(3) ≈ 0.63`。

In [ ]:
# ✅ 跑一下：训练若干轮，再在 validation 上评估
# CPU 上会明显慢，可以先翻上面的输出；每轮都会打印当前进度
import time

# 上面那格 probe 已经从 train_loader 取走了 30 个 batch，生成器状态往前走过一段。
# 正式实验前重置随机状态并重建 loader，结果才不会随「probe 跑没跑、跑了几次」漂移。
labkit.set_seed(42)
train_loader = labkit.create_train_loader(data.train, batch_size=256, seed=42)

model = TwoTower(data.num_users, data.num_items, embedding_dim=64, temperature=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)   # 1e-3 在这个结构上 5 轮学不动

EPOCHS = 5
started = time.time()
for epoch in range(1, EPOCHS + 1):
    epoch_loss = train_one_epoch(model, train_loader, optimizer)
    print(
        f'epoch {epoch}/{EPOCHS}  平均 loss = {epoch_loss:.4f}'
        f'  已用 {time.time() - started:.0f}s'
    )

# 正式评估还要过滤此前播放过但未达正反馈阈值的物品，labkit 已按目标时间构造。
seen_items = data.validation_seen
validation_users = [row.user_idx for row in data.validation]
recommendations = recommend_top_k(
    model, validation_users, data.num_items, seen_items, k=20
)

print()
for k in (10, 20):
    print(
        f'Recall@{k} = {recall_at_k(recommendations, data.validation, k):.6f}'
        f'   NDCG@{k} = {ndcg_at_k(recommendations, data.validation, k):.6f}'
    )

baseline_recall20 = recall_at_k(recommendations, data.validation, 20)

# 把这次「不加 mask」的基线记下来。Lab 3 会用同一套配置再跑一遍并比对，
# 用来确认两章确实用的是同样的 seed、loader 和训练超参。
labkit.record('no_mask_recall20', baseline_recall20)
print()
print('已记录不加 mask 的基线 Recall@20 =', f'{baseline_recall20:.6f}')

check('2-5 每个评测用户都有推荐', len(recommendations), len(validation_users))
check('2-5 前 200 个用户的推荐里没有已看物品',
      [user for user in validation_users[:200]
       if set(recommendations[user]) & seen_items.get(user, set())],
      [])
check('2-5 Recall@20 落在 [0, 1]',
      0.0 <= recall_at_k(recommendations, data.validation, 20) <= 1.0,
      True)

## 把上面两行放在一起看

loss 从 6.15 一路降到 4.76，而 Recall@20 停在 0.079339——3,025 个评测用户里约 7.9% 的人，下一次点开的电影进了前 20 名。

（正文里同一位置大约是 0.048，那是 MLP tower + lr=0.001 的配置。两边都是真跑出来的结果，配置不同，数字不能直接横向比。）

这两个数不矛盾，它们量的不是同一件事。训练时每一行只要在 256 个候选里把正确那个排到最前；评估时是从 3,533 部电影里挑前 20，命中他真正点开的那一部。

顺便记一个坑：这个结构的学习率如果取 1e-3，5 轮之后 Recall@20 会停在 0.006 附近——正好是随机排序的期望值（20 / 3533）。模型其实什么都没学到，但 loss 看上去还在缓慢下降。

那还有没有别的原因让 Recall 这么低？有。训练时每一行是把同批其他用户的物品当作负样本的，可其中有些物品，本来就是这一行那个用户在训练集里喜欢过的东西——目标在这里自相矛盾。

这个问题下一章会量出来，先不猜。

In [ ]:
# 💾 导出这一章写的东西，Lab 3 直接 import
labkit.harvest(
    'myrec/train.py',
    train_one_epoch,
    imports=['import torch', 'from myrec.model import train_one_step'],
)
labkit.harvest(
    'myrec/eval.py',
    group_seen_items,
    recommend_top_k,
    recall_at_k,
    ndcg_at_k,
    imports=['import math', 'from collections import defaultdict', 'import torch'],
)

# 顺手确认一下导出能用
import importlib

import myrec.eval as my_eval
import myrec.train as my_train

importlib.reload(my_train)
importlib.reload(my_eval)

check(
    '2-6 交接成功：myrec.train 和 myrec.eval 都能用',
    [hasattr(my_train, 'train_one_epoch'), hasattr(my_eval, 'recommend_top_k')],
    [True, True],
)

## 收尾

### 这个 lab 写了多少

| 自己写的 | 行数 | 脚手架给的 |
|---|---|---|
| `train_one_epoch` | 6 | `labkit.py` 的数据管道 |
| `group_seen_items` | 4 | `check()` / `check_close()` |
| `recommend_top_k` | 18 | 接 Lab 1 的 `myrec/model.py` |
| `recall_at_k` / `ndcg_at_k` | 13 | 真实数据与 DataLoader |

加起来 40 行左右。到这里，训练和评估两条链路都通了。

### 这几条应该能直接答上来

1. `train_one_epoch` 为什么不用再写一遍 `zero_grad` / `backward` / `step`？
2. 评估前为什么要把已看物品的分数压成 `-inf`？不压会怎样？
3. 为什么 item 向量可以一次性算完，用户向量却要分批？
4. 单 target 下 NDCG 的分母为什么可以省掉？
5. 训练 loss 和 Recall@K 分别回答什么问题？

答着不顺就回头看对应的那几格。

### 还没解决的问题

上面的候选集里，有多少位置其实是这个用户已经喜欢过、却被当成负样本的物品？

下一章会用你自己写的模型和评估，把它量出来，然后只改训练目标，看 Recall 会动多少。

### 下一个 lab

**Lab 3｜false negative 诊断与 positive mask**：写一个诊断脚本统计 batch 内的冲突比例，把已知正样本从 CrossEntropy 的候选里屏蔽掉，再用同一套评估做单变量对照。